# Physics-informed modeling (critical-point penalty + monotonicity)

On this notebook we're going to enforce to constraints to the surface tension: "`Surface_tension` -> 0 as `T` -> `TC`" as a **
collocation penalty**: synthetic near-TC rows are added to training, target forced toward 0,
and `Surface_tension` must never increase with
`T`, anywhere in the training range - not just near `TC`.

`TC` is never a model feature and is never required at `predict()` time - it's only ever
forwarded as an explicit fit-param for building collocation rows, and consumed post-hoc to audit
predictions.

# Importing train/test datasets and building X, y

In [ ]:
import copy, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from scipy.stats import randint, uniform

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error, root_mean_squared_error,
                             r2_score)
from sklearn.utils.validation import check_is_fitted
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor

SPLITS = ['Family']
TARGET = 'Surface_tension'

with open(r"Data and variables/GC_variables.pkl", "rb") as f:
    group_cols = pickle.load(f)["group_cols"]          
with open(r"Data and variables/ML_variables.pkl", "rb") as f:
    descriptor_cols = pickle.load(f)["descriptor_cols"] 

# Only the GC (group-contribution) feature set is used here - it's half of the best-performing
# combination (Family split / GC) found in the original sweep, so it's the best starting point to try to
# push violations down even further.
FEATURES = {'GC': group_cols, 'Descriptors': descriptor_cols + ['T']}
FSETS = list(FEATURES)

# Same logic used in 02_ChemicalDataset to build each train/test split:
# StratifiedGroupKFold with groups=ChemID, stratified by chemical Family. CV folds are needed again here
# for RandomizedSearchCV (see the Helper functions / Models cells below).
STRAT_COL = {'Family': 'Family'}
GROUP_COL = 'ChemID'
N_CV = 5

def make_cv(train, split, n_splits=N_CV):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')     # singleton clusters/families can't be stratified
        folds = list(sgkf.split(train, y=train[STRAT_COL[split]], groups=train[GROUP_COL]))
    for tr, va in folds:
        assert set(train.iloc[tr][GROUP_COL]).isdisjoint(train.iloc[va][GROUP_COL])
    return folds

data = {}
train_TC = {}
test_meta = {}
for s in SPLITS:
    train = pd.read_pickle(rf'Data and variables/train_{s}.pkl')
    test = pd.read_pickle(rf'Data and variables/test_{s}.pkl')
    cv = make_cv(train, s)
    train_TC[s] = train['TC'].reset_index(drop=True)
    test_meta[s] = test[['ChemID', 'Family', 'TC']]
    for fs, cols in FEATURES.items():
        data[(fs, s)] = dict(X_train=train[cols].reset_index(drop=True),
                             y_train=train[TARGET].reset_index(drop=True),
                             X_test=test[cols], y_test=test[TARGET], cv=cv)

# the models cannot handle missing/infinite values
for (fs, s), d in data.items():
    for part in ('X_train', 'X_test'):
        bad = ~np.isfinite(d[part].to_numpy(dtype=float))
        assert not bad.any(), f"{fs}/{s}/{part}: NaN/inf in columns {list(d[part].columns[bad.any(axis=0)])}"
    print(fs, s, d['X_train'].shape, d['X_test'].shape)

# metrics table (saved SEPARATELY from 03/04/05/06's metrics pickles), fitted models and per-fold CV
# metrics of the best candidate found by RandomizedSearchCV
COLUMNS = ['description', 'algorithm', 'MSE', 'RMSE', 'MAE', 'SD_error', 'NRMSE_%', 'WAPE_%', 'r2_score']
metrics_paths = {s: rf'Data and variables/metrics_{s}_physics_monotonic.pkl' for s in SPLITS}
metrics = {}
for s in SPLITS:
    try:
        metrics[s] = pd.read_pickle(metrics_paths[s])
    except FileNotFoundError:
        metrics[s] = pd.DataFrame(columns=COLUMNS)

models = {}      # (algorithm, feature_set, split) -> fitted estimator
cv_folds = {}     # (algorithm, feature_set, split) -> per-fold validation metrics of the best candidate

# Helper functions

Same evaluation/plotting/audit helpers as `06`. Two ways to fit a model are available:
- `fit_and_record` - fit once with fixed, already-chosen hyperparameters (no search). Used for the MLP,
  whose best hyperparameters were already found in `06` and aren't being re-searched here.
- `fit_all`/`get_best` - `06`'s original `RandomizedSearchCV`-based search, grouped CV (`StratifiedGroupKFold`
  by `ChemID`). Use this for `PhysicsConstrainedRegressor`-wrapped estimators (XGBoost or anything else
  with a `sample_weight`-aware `.fit()`) whenever you want to tune hyperparameters - including the
  monotonic variants, since `monotone_constraints` is just a fixed argument to the wrapped estimator and
  doesn't stop everything else from being searched.

In [ ]:
def get_metrics(model, X_test, y_test, metrics, path, model_desc, algorithm):
    """Evaluate on the test set, add/replace the row in `metrics`, save it and return it."""
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred
    rmse = root_mean_squared_error(y_test, y_pred)

    row = pd.DataFrame([{
        'description': model_desc,
        'algorithm': algorithm,
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': rmse,
        'MAE': mean_absolute_error(y_test, y_pred),
        'SD_error': np.std(residuals, ddof=1),
        'NRMSE_%': rmse / np.mean(y_test) * 100,
        'WAPE_%': residuals.abs().sum() / y_test.abs().sum() * 100,
        'r2_score': r2_score(y_test, y_pred),
    }])

    keep = ~((metrics['description'] == model_desc) & (metrics['algorithm'] == algorithm))
    metrics = pd.concat([metrics[keep], row], ignore_index=True)
    metrics.to_pickle(path)
    return metrics


def plot_actual_vs_predicted(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.text(0.05, 0.95, f'{title}\nR² = {r2_score(y_test, y_pred):.3f}',
             transform=plt.gca().transAxes, va='top', bbox=dict(facecolor='white', alpha=0.8))
    plt.show()


def plot_training_curves(algorithm, fs, split, title):
    """Loss and R2 per epoch for a fitted Torch MLP, from its training history."""
    net = models[(algorithm, fs, split)]
    h = net.history_

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h['epoch'], h['train_loss'], label='train')
    axes[0].plot(h['epoch'], h['val_loss'], label='val')
    axes[0].axvline(net.best_epoch_, color='gray', linestyle='--', label='best epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE loss (standardized y, data term only)')
    axes[0].set_title(f'Loss vs Epoch - {title}')
    axes[0].legend()

    axes[1].plot(h['epoch'], h['train_r2'], label='train')
    axes[1].plot(h['epoch'], h['val_r2'], label='val')
    axes[1].axvline(net.best_epoch_, color='gray', linestyle='--', label='best epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('R²')
    axes[1].set_title(f'R² vs Epoch - {title}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def fit_and_record(label, estimator, fs='GC', s='Family', plot=True):
    """Fit once (no search - estimator already carries its final hyperparameters), evaluate on the test
    set and store the result under `label`. TC is always forwarded as a fit-param - every estimator used
    in this notebook is physics-capable (either a PhysicsConstrainedRegressor or a Torch MLP)."""
    d = data[(fs, s)]
    print(f'--- {label} | {fs} | {s} split ---')
    model = estimator.fit(d['X_train'], d['y_train'], TC=train_TC[s])
    models[(label, fs, s)] = model
    metrics[s] = get_metrics(model, d['X_test'], d['y_test'], metrics[s], metrics_paths[s], fs, label)
    if plot:
        plot_actual_vs_predicted(model, d['X_test'], d['y_test'], f'{label} ({fs}, {s})')
    return model


CV_SCORING = {'RMSE': 'neg_root_mean_squared_error', 'MAE': 'neg_mean_absolute_error', 'r2': 'r2',
              'MSE': 'neg_mean_squared_error'}


def get_best(reg, params, X, y, cv, TC=None, n_iter=50, n_jobs=-1):
    """Randomized search selecting on MSE. If `TC` is given it's forwarded as a fit_param - scikit-learn
    slices it per CV fold by position, exactly like X/y, so a fold never sees another fold's TC values,
    and it's never part of X. Returns (best estimator, per-fold metrics of the best candidate)."""
    rcv = RandomizedSearchCV(reg, params, n_iter=n_iter, cv=cv, scoring=CV_SCORING, refit='MSE',
                             random_state=42, n_jobs=n_jobs)
    fit_kwargs = {} if TC is None else {'TC': TC}
    rcv.fit(X, y, **fit_kwargs)
    res, i = rcv.cv_results_, rcv.best_index_
    n_folds = rcv.n_splits_
    folds = pd.DataFrame({
        'fold': range(1, n_folds + 1),
        'n_val': [len(v) for _, v in cv],
        'RMSE': [-res[f'split{k}_test_RMSE'][i] for k in range(n_folds)],
        'MAE': [-res[f'split{k}_test_MAE'][i] for k in range(n_folds)],
        'r2': [res[f'split{k}_test_r2'][i] for k in range(n_folds)],
    }).set_index('fold')
    print('best params:', rcv.best_params_)
    return rcv.best_estimator_, folds


def show_folds(folds, title):
    summary = pd.DataFrame({'mean': folds[['RMSE', 'MAE', 'r2']].mean(), 'std': folds[['RMSE', 'MAE', 'r2']].std()}).T
    print(f'CV metrics per fold - {title}')
    display(pd.concat([folds, summary]).round(3))


def prefix_params(params, prefix):
    """Nest a flat param grid under a meta-estimator's sub-estimator, e.g. for
    PhysicsConstrainedRegressor(estimator=...): 'max_depth' -> 'estimator__max_depth'."""
    return {f'{prefix}{k}': v for k, v in params.items()}


def fit_all(label, build_reg, params, fsets=FSETS, splits=SPLITS, n_iter=50, plot=True, n_jobs=-1):
    """For every feature set/split given: tune a fresh `build_reg(fs)` with randomized CV (grouped by
    ChemID, stratified like the split), evaluate on the test set and store the results under `label`.
    `build_reg` takes the feature-set name so e.g. a monotonic constraint tuple (which depends on that
    feature set's column order) can be rebuilt per feature set. TC is always forwarded as a fit_param -
    every estimator passed to `fit_all` is expected to be physics-capable (a PhysicsConstrainedRegressor,
    here always wrapping something that also accepts `TC` as a fit-param)."""
    for fs in fsets:
        for s in splits:
            print(f'--- {label} | {fs} | {s} split ---')
            d = data[(fs, s)]
            best, folds = get_best(build_reg(fs), params, d['X_train'], d['y_train'], cv=d['cv'],
                                   TC=train_TC[s], n_iter=n_iter, n_jobs=n_jobs)
            cv_folds[(label, fs, s)] = folds
            show_folds(folds, f'{label} ({fs}, {s})')
            models[(label, fs, s)] = best
            metrics[s] = get_metrics(best, d['X_test'], d['y_test'], metrics[s], metrics_paths[s], fs, label)
            if plot:
                plot_actual_vs_predicted(best, d['X_test'], d['y_test'], f'{label} ({fs}, {s})')


def check_tc_violations(algorithm, fs, split, tr_thresholds=(0.90, 0.95, 0.98, 0.99), zero_tol=2.0, worst_n=15):
    """Physical sanity check on the TEST set only: Surface_tension must go to ~0 as T approaches the
    critical temperature TC. A "violation" is a near-TC test row where the model still predicts a
    non-negligible value (> zero_tol mN/m)."""
    d = data[(fs, split)]
    model = models[(algorithm, fs, split)]
    meta = test_meta[split]
    y_pred = model.predict(d['X_test'])
    title = f'{algorithm} ({fs}, {split})'

    detail = pd.DataFrame({
        'ChemID': meta['ChemID'].values,
        'Family': meta['Family'].values,
        'T': d['X_test']['T'].values,
        'TC': meta['TC'].values,
        'actual': d['y_test'].values,
        'predicted': y_pred,
    })
    detail['Tr'] = detail['T'] / detail['TC']

    summary = pd.DataFrame([
        {
            'T/TC >=': thr,
            'n_rows': int((detail['Tr'] >= thr).sum()),
            'n_violations': int(((detail['Tr'] >= thr) & (detail['predicted'] > zero_tol)).sum()),
        }
        for thr in tr_thresholds
    ])
    print(f'Critical-point violations (predicted Surface_tension > {zero_tol} mN/m near TC) - {title}')
    display(summary)

    near_tc = detail[detail['Tr'] >= min(tr_thresholds)]
    worst = near_tc[near_tc['predicted'] > zero_tol].sort_values('Tr', ascending=False)
    if len(worst):
        print(f'{min(len(worst), worst_n)} worst violations (closest to TC, by T/TC) - {title}')
        display(worst.head(worst_n).round(3))
    else:
        print(f'No violations found at the loosest threshold tested (T/TC >= {min(tr_thresholds)}).')

    return detail


def tc_violation_counts(algorithm, fs, split, tr_thresholds=(0.90, 0.95, 0.98, 0.99), zero_tol=2.0):
    """Same violation definition as check_tc_violations, but just the counts - for the comparison table."""
    d = data[(fs, split)]
    model = models[(algorithm, fs, split)]
    meta = test_meta[split]
    y_pred = model.predict(d['X_test'])
    Tr = d['X_test']['T'].values / meta['TC'].values
    return pd.DataFrame([
        {'algorithm': algorithm, 'feature_set': fs, 'split': split, 'T/TC >=': thr,
         'n_rows': int((Tr >= thr).sum()),
         'n_violations': int(((Tr >= thr) & (y_pred > zero_tol)).sum())}
        for thr in tr_thresholds
    ])

# Physics-informed building blocks

In [ ]:
TC_TR_LEVELS = (0.99, 1.0)

def make_collocation(X, TC, t_col='T', tr_levels=TC_TR_LEVELS):
    """One synthetic row per (real row, tr_level): same structural features, T replaced by
    tr_level * TC. A handful of compounds have no recorded TC - those rows are excluded here (they still
    train normally on the real data loss, they just can't contribute a collocation constraint)."""
    mask = TC.notna().to_numpy()
    X, TC = X.loc[mask].reset_index(drop=True), TC.loc[mask].reset_index(drop=True)
    rows = []
    for tr in tr_levels:
        x = X.copy()
        x[t_col] = tr * TC.to_numpy()
        rows.append(x)
    return pd.concat(rows, ignore_index=True)

### Searching a per-tr-level `collocation_weight`

`PhysicsConstrainedRegressor.collocation_weight` already accepts a `{tr_level: weight}` dict (see its
docstring above), but `RandomizedSearchCV` can't search a dict directly - it needs something with an
`.rvs(random_state=...)` method, the same interface `scipy.stats` distributions expose. `CollocationWeightDist`
provides that: it samples a base weight for the looser levels (e.g. `tr=0.99`) and a `ratio >= 1` applied
only to the last (closest-to-TC) level in `tr_levels` (`tr=1.0` here), so the exact-TC boundary rows are
always weighted at least as heavily as the others, never less.

In [ ]:
class CollocationWeightDist:
    """RandomizedSearchCV-compatible distribution over per-tr-level collocation_weight dicts. Every level
    except the last gets a shared `base` weight (sampled from `base_dist`); the last tr_level (tr=1.0 -
    the exact critical point, with TC_TR_LEVELS=(0.99, 1.0)) gets `base * ratio`, with `ratio` sampled
    from `ratio_dist` and `ratio_low >= 1.0` so that level is never weighted less than the others."""

    def __init__(self, base_dist=uniform(0.2, 1), ratio_dist=uniform(1.0, 2.0), tr_levels=TC_TR_LEVELS):
        self.base_dist = base_dist
        self.ratio_dist = ratio_dist
        self.tr_levels = tr_levels

    def rvs(self, random_state=None):
        base = self.base_dist.rvs(random_state=random_state)
        ratio = self.ratio_dist.rvs(random_state=random_state)
        weights = {tr: float(base) for tr in self.tr_levels}
        weights[self.tr_levels[-1]] = float(base * ratio)
        return weights


### XGBoost - collocation penalty (sample-weighted) + native monotonicity

`PhysicsConstrainedRegressor` is unchanged from `06`: it wraps any estimator whose `.fit()` accepts
`sample_weight`, appending near-TC collocation rows (target 0, weighted by `collocation_weight`) before
delegating to the wrapped estimator.

Monotonicity is added separately, as a native **hard constraint** on the wrapped `XGBRegressor` itself:
`monotone_constraints` takes a tuple, one entry per feature column in `X`'s column order
(`-1`=non-increasing, `0`=unconstrained, `+1`=non-decreasing). `build_monotonic_cst` sets `-1` at `T`'s
position and `0` everywhere else, so the ensemble's splits are constrained such that `Surface_tension` can
never increase as `T` increases - combined with, not instead of, the collocation penalty above.

In [ ]:
class PhysicsConstrainedRegressor(BaseEstimator, RegressorMixin):
    """`collocation_weight` is either a single scalar (same weight for every collocation row, as before)
    or a mapping {tr_level: weight} / sequence matching `tr_levels` in order, so the penalty can be
    pushed harder on the rows closer to the critical point (e.g. tr=1.0, the exact TC boundary) than on
    the looser ones (e.g. tr=0.99) without changing anything else about the collocation scheme."""

    def __init__(self, estimator=None, t_col='T', tr_levels=TC_TR_LEVELS, collocation_weight=0.3):
        self.estimator = estimator
        self.t_col = t_col
        self.tr_levels = tr_levels
        self.collocation_weight = collocation_weight

    def _weights_per_level(self):
        w = self.collocation_weight
        if np.isscalar(w):
            return [w] * len(self.tr_levels)
        if isinstance(w, dict):
            return [w[tr] for tr in self.tr_levels]
        w = list(w)
        assert len(w) == len(self.tr_levels), \
            "collocation_weight must be scalar, a {tr: weight} dict, or match len(tr_levels)"
        return w

    def fit(self, X, y, TC=None, sample_weight=None):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y)).reset_index(drop=True)
        TC = pd.Series(np.asarray(TC)).reset_index(drop=True)

        n_valid = int(TC.notna().sum())   # rows per tr_level block - make_collocation drops TC-less rows once per level
        X_colloc = make_collocation(X, TC, self.t_col, self.tr_levels)
        y_colloc = pd.Series(0.0, index=X_colloc.index)
        w_real = np.ones(len(X)) if sample_weight is None else np.asarray(sample_weight)
        w_colloc = np.concatenate([np.full(n_valid, w) for w in self._weights_per_level()])

        X_all = pd.concat([X, X_colloc], ignore_index=True)
        y_all = pd.concat([y, y_colloc], ignore_index=True)
        w_all = np.concatenate([w_real, w_colloc])

        self.estimator_ = clone(self.estimator).fit(X_all, y_all, sample_weight=w_all)
        return self

    def predict(self, X):
        check_is_fitted(self, 'estimator_')
        return self.estimator_.predict(X)   # TC never needed/used here


def build_monotonic_cst(cols, t_col='T'):
    """All-zero (unconstrained) except -1 at t_col's position: Surface_tension cannot increase with T."""
    constraints = [0] * len(cols)
    constraints[list(cols).index(t_col)] = -1
    return tuple(constraints)

### MLP (PyTorch) - gradient-based collocation penalty + soft monotonicity penalty

Same `[Linear -> BatchNorm -> ReLU -> Dropout]` architecture and TC-penalty mechanism as `06`'s
`TorchPhysicsInformedMLP`. A second, independent penalty is added: `monotonic_weight` controls a soft
squared-hinge term computed by evaluating each training batch twice - once as-is, once with `T` nudged up
by `monotonic_eps` **in standardized units** (so no real-world T scale/units need to be known here) - and
penalizing any *increase* in predicted `Surface_tension` from that nudge. `monotonic_weight=0.0` disables
it and reduces this class to `06`'s original (used for the `_physics`-only replica below, so the
comparison is a controlled ablation: identical architecture/weights search, monotonicity term on vs.
off). As with the TC penalty, this is a soft training-time nudge, not a hard shape guarantee - `predict()`
is unaffected and never needs TC.

In [ ]:
class TorchMonotonicPhysicsInformedMLP(BaseEstimator, RegressorMixin):
    def __init__(self, hidden_layer_sizes=(64, 32), dropout=0.2, weight_decay=1e-4, lr=1e-3,
                 batch_size=64, max_epochs=200, patience=15, val_size=0.15,
                 lambda_physics=1.0, zero_tol=ZERO_TOL, tr_levels=TC_TR_LEVELS,
                 monotonic_weight=0.0, monotonic_eps=0.05, t_col='T',
                 random_state=42, device=None, progress_bar=True):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.weight_decay = weight_decay
        self.lr = lr
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.val_size = val_size
        self.lambda_physics = lambda_physics
        self.zero_tol = zero_tol
        self.tr_levels = tr_levels
        self.monotonic_weight = monotonic_weight
        self.monotonic_eps = monotonic_eps
        self.t_col = t_col
        self.random_state = random_state
        self.device = device
        self.progress_bar = progress_bar

    def _build(self, n_features):
        device = self.device or ('cuda' if torch.cuda.is_available() else 'cpu')
        layers, in_dim = [], n_features
        for h in self.hidden_layer_sizes:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(self.dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        return nn.Sequential(*layers).to(device), device

    def fit(self, X, y, TC=None):
        torch.manual_seed(self.random_state)
        X = pd.DataFrame(X).reset_index(drop=True)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        TC = pd.Series(np.asarray(TC, dtype=np.float32)).reset_index(drop=True)

        self.feature_names_ = list(X.columns)
        t_idx = self.feature_names_.index(self.t_col)

        self.x_scaler_ = StandardScaler().fit(X.to_numpy(dtype=np.float32))
        self.y_scaler_ = StandardScaler().fit(y)

        Xs = self.x_scaler_.transform(X.to_numpy(dtype=np.float32))
        ys = self.y_scaler_.transform(y)

        X_tr, X_val, y_tr, y_val, TC_tr, _ = train_test_split(
            Xs, ys, TC, test_size=self.val_size, random_state=self.random_state)

        self.model_, self.device_ = self._build(Xs.shape[1])
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        mse = nn.MSELoss()

        X_tr_t = torch.tensor(X_tr, device=self.device_)
        y_tr_t = torch.tensor(y_tr, device=self.device_)
        X_val_t = torch.tensor(X_val, device=self.device_)
        y_val_t = torch.tensor(y_val, device=self.device_)

        valid = ~np.isnan(TC_tr.to_numpy())
        X_tr_raw = self.x_scaler_.inverse_transform(X_tr)[valid]
        TC_tr_valid = TC_tr.to_numpy()[valid]
        X_colloc_t = None
        if len(X_tr_raw):
            colloc_raw = []
            for tr in self.tr_levels:
                x = X_tr_raw.copy()
                x[:, t_idx] = tr * TC_tr_valid
                colloc_raw.append(x)
            X_colloc_t = torch.tensor(self.x_scaler_.transform(np.concatenate(colloc_raw, axis=0)),
                                      dtype=torch.float32, device=self.device_)

        y_mean, y_std = float(self.y_scaler_.mean_[0]), float(self.y_scaler_.scale_[0])

        def physics_penalty(pred_std):
            pred_real = pred_std * y_std + y_mean
            return torch.relu(pred_real.abs() - self.zero_tol).pow(2).mean()

        def monotonic_penalty(X_batch):
            """Nudge T up by monotonic_eps (standardized units) and penalize any resulting increase in
            the (real-unit) prediction - Surface_tension must not increase with T anywhere."""
            X_pert = X_batch.clone()
            X_pert[:, t_idx] = X_pert[:, t_idx] + self.monotonic_eps
            pred_real = self.model_(X_batch) * y_std + y_mean
            pred_pert_real = self.model_(X_pert) * y_std + y_mean
            return torch.relu(pred_pert_real - pred_real).pow(2).mean()

        rng = np.random.default_rng(self.random_state)
        best_val, best_state, best_optimizer_state, best_epoch, stall, epoch = np.inf, None, None, 0, 0, 0
        history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_r2': [], 'val_r2': []}

        pbar = tqdm(range(self.max_epochs), desc='MLP training', disable=not self.progress_bar, leave=False)
        for epoch in pbar:
            self.model_.train()
            perm = rng.permutation(len(X_tr_t))
            for start in range(0, len(perm), self.batch_size):
                idx = perm[start:start + self.batch_size]
                if len(idx) < 2:
                    continue   # BatchNorm needs >=2 samples per batch in train mode
                idx_t = torch.as_tensor(idx, dtype=torch.long, device=self.device_)
                opt.zero_grad()
                X_batch = X_tr_t[idx_t]
                pred = self.model_(X_batch)
                loss = mse(pred, y_tr_t[idx_t])
                if self.lambda_physics and X_colloc_t is not None:
                    loss = loss + self.lambda_physics * physics_penalty(self.model_(X_colloc_t))
                if self.monotonic_weight:
                    loss = loss + self.monotonic_weight * monotonic_penalty(X_batch)
                loss.backward()
                opt.step()

            self.model_.eval()
            with torch.no_grad():
                train_pred = self.model_(X_tr_t)
                val_pred = self.model_(X_val_t)
                train_loss = mse(train_pred, y_tr_t).item()
                val_loss = mse(val_pred, y_val_t).item()

            history['epoch'].append(epoch)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_r2'].append(r2_score(y_tr, train_pred.cpu().numpy()))
            history['val_r2'].append(r2_score(y_val, val_pred.cpu().numpy()))

            if val_loss < best_val - 1e-6:
                best_val, best_epoch, stall = val_loss, epoch, 0
                best_state = {k: v.clone() for k, v in self.model_.state_dict().items()}
                best_optimizer_state = copy.deepcopy(opt.state_dict())
            else:
                stall += 1
            pbar.set_postfix(train_loss=f'{train_loss:.4f}', val_loss=f'{val_loss:.4f}',
                             best_val=f'{best_val:.4f}', stall=stall)
            if stall >= self.patience:
                break
        pbar.close()

        self.model_.load_state_dict(best_state)
        self.optimizer_state_dict_ = best_optimizer_state
        self.n_epochs_, self.best_val_loss_, self.best_epoch_ = epoch + 1, best_val, best_epoch
        self.history_ = history
        return self

    def predict(self, X):
        check_is_fitted(self, 'model_')
        X = pd.DataFrame(X)[self.feature_names_].to_numpy(dtype=np.float32)
        Xs = self.x_scaler_.transform(X)
        self.model_.eval()
        with torch.no_grad():
            pred_std = self.model_(torch.tensor(Xs, device=self.device_)).cpu().numpy()
        return self.y_scaler_.inverse_transform(pred_std).ravel()

# Models

Fixed hyperparameters only - the exact best values `RandomizedSearchCV` already found in `06` for
GC/Family, reused as-is. No re-search; only the monotonicity mechanism is varied across the `_physics`
(TC penalty alone, a straight replica of `06`'s best model, for reference) vs. `_physics_monotonic`
(TC penalty + monotonicity) pair for each algorithm.

In [ ]:
XGB_PARAMS = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(2, 10),
    'learning_rate': uniform(0.01, 0.29),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(0.5, 3),
}
# collocation_weight is searched as a per-tr-level dict via CollocationWeightDist (see the "Searching a
# per-tr-level collocation_weight" cell above): tr=1.0 (exact TC) always gets >= as much weight as tr=0.99.

fit_all('XGBoost_physics',
        lambda fs: PhysicsConstrainedRegressor(estimator=XGBRegressor(objective='reg:squarederror',
                                                                      random_state=42, n_jobs=1)),
        {**prefix_params(XGB_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

fit_all('XGBoost_physics_monotonic',
        lambda fs: PhysicsConstrainedRegressor(
            estimator=XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=1,
                                   monotone_constraints=build_monotonic_cst(FEATURES[fs]))),
        {**prefix_params(XGB_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

# HistGradientBoostingRegressor: also sample_weight-aware, and - like RandomForestRegressor below -
# supports the same monotonic_cst convention as XGBoost's monotone_constraints (-1/0/+1 per feature, in
# column order), so both a _physics and a _physics_monotonic variant are run, same pattern as XGBoost
# above. Note its hyperparameters differ from XGBoost's (max_iter instead of n_estimators, no subsample).
HGB_PARAMS = {
    'max_iter': randint(100, 800),
    'max_depth': randint(2, 12),
    'learning_rate': uniform(0.01, 0.29),
    'max_leaf_nodes': randint(15, 63),
    'min_samples_leaf': randint(5, 50),
    'l2_regularization': uniform(0, 2),
}

fit_all('GradientBoosting_physics',
        lambda fs: PhysicsConstrainedRegressor(estimator=HistGradientBoostingRegressor(random_state=42)),
        {**prefix_params(HGB_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

fit_all('GradientBoosting_physics_monotonic',
        lambda fs: PhysicsConstrainedRegressor(
            estimator=HistGradientBoostingRegressor(random_state=42,
                                                    monotonic_cst=build_monotonic_cst(FEATURES[fs]))),
        {**prefix_params(HGB_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

# RandomForestRegressor: also sample_weight-aware, and (since scikit-learn 1.4) supports the same
# monotonic_cst convention as XGBoost's monotone_constraints (-1/0/+1 per feature, in column order), so
# both a _physics and a _physics_monotonic variant are run, same pattern as XGBoost above.
RF_PARAMS = {
    'n_estimators': randint(100, 800),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': uniform(0.3, 0.7),
}

fit_all('RandomForest_physics',
        lambda fs: PhysicsConstrainedRegressor(estimator=RandomForestRegressor(random_state=42, n_jobs=1)),
        {**prefix_params(RF_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

fit_all('RandomForest_physics_monotonic',
        lambda fs: PhysicsConstrainedRegressor(
            estimator=RandomForestRegressor(random_state=42, n_jobs=1,
                                            monotonic_cst=build_monotonic_cst(FEATURES[fs]))),
        {**prefix_params(RF_PARAMS, 'estimator__'), 'collocation_weight': CollocationWeightDist()},
        fsets=['GC'], n_iter=20)

# To try yet another PhysicsConstrainedRegressor-wrapped algorithm, copy the pattern above: a flat param
# grid for the estimator's own hyperparameters, prefixed with prefix_params(..., 'estimator__'), plus
# 'collocation_weight': CollocationWeightDist() for the (per-tr-level) penalty strength, and (optionally)
# a native monotonicity constructor arg (name/convention varies by estimator) if it supports one.

In [ ]:
MLP_BEST_PARAMS = dict(
    hidden_layer_sizes=(128, 64, 32), dropout=0.18727005942368125,
    lr=0.0036256176345762273, weight_decay=0.00024400607090817574,
)
MLP_LAMBDA_PHYSICS = 0.0054167545832474575

fit_and_record('MLP_PyTorch_physics',
               TorchMonotonicPhysicsInformedMLP(max_epochs=200, patience=15,
                                                lambda_physics=MLP_LAMBDA_PHYSICS,
                                                monotonic_weight=0.0, **MLP_BEST_PARAMS))
plot_training_curves('MLP_PyTorch_physics', 'GC', 'Family', 'MLP_PyTorch_physics (GC, Family)')

fit_and_record('MLP_PyTorch_physics_monotonic',
               TorchMonotonicPhysicsInformedMLP(max_epochs=200, patience=15,
                                                lambda_physics=MLP_LAMBDA_PHYSICS,
                                                monotonic_weight=1.0, **MLP_BEST_PARAMS))
plot_training_curves('MLP_PyTorch_physics_monotonic', 'GC', 'Family',
                     'MLP_PyTorch_physics_monotonic (GC, Family)')

# Comparison

For a comparable test-set RMSE/MAE/r2, does adding the monotonicity constraint reduce near-TC
violations further than the TC-penalty-only baseline?

In [ ]:
comparison = metrics['Family'][metrics['Family']['algorithm'].isin(ALGORITHMS)]
display(comparison.reset_index(drop=True))

### Critical-point violations

In [ ]:
ALGORITHMS = ['XGBoost_physics','XGBoost_physics_monotonic',
              'GradientBoosting_physics','GradientBoosting_physics_monotonic',
              'RandomForest_physics','RandomForest_physics_monotonic']

In [ ]:
rows = [tc_violation_counts(alg, 'GC', 'Family') for alg in ALGORITHMS]
violations = pd.concat(rows, ignore_index=True)
pivot = violations.pivot_table(index='T/TC >=', columns='algorithm', values='n_violations')
display(pivot[ALGORITHMS])

### Detailed audit

In [ ]:
for alg in ALGORITHMS:
    check_tc_violations(alg, 'GC', 'Family')

### Takeaway

Fill in after running: did adding the monotonicity constraint (native hard constraint for XGBoost, soft
gradient penalty for the MLP) reduce critical-point violations further than the TC-penalty-only baseline,
for a comparable test RMSE/MAE/r2? Note whether violations were **eliminated** (`n_violations == 0` at
every threshold) or only **reduced further** - for XGBoost the shape constraint is hard, but still doesn't
force an exact zero at TC by construction; for the MLP, monotonicity is only a soft training-time nudge
(like the TC penalty itself), so some residual non-monotonicity or near-TC violation is possible even in
the best case.